# A Neural-Network Framework Implemented with NumPy

This notebook demonstrates a compact neural-network framework built without PyTorch or TensorFlow. The accompanying `modules` package implements forward and backward passes, trainable parameters, normalization, regularization, losses, optimizers, and mini-batch iteration directly with NumPy.

The project originated as coursework for the HSE Faculty of Computer Science and was reorganized into a standalone portfolio project. The implementation and experiments shown here are the author's own work.

## Implemented components

- affine and low-rank linear layers;
- ReLU, Sigmoid, GELU, Softmax, and LogSoftmax activations;
- Batch Normalization and Dropout with distinct train/evaluation behaviour;
- a Sequential container with recursive parameter and gradient handling;
- MSE and cross-entropy losses with label smoothing;
- SGD with momentum/Nesterov updates and Adam;
- a mini-batch DataLoader.

The source code is available in the adjacent `modules/` directory.

## End-to-end regression check

The following experiment trains the framework on a synthetic non-linear regression problem,

$$y=\sum_j \sin(x_j).$$

It verifies that the individual components work together in a complete forward/backward optimization loop.

In [ ]:
import numpy as np
import modules as mm
from tqdm.notebook import tqdm

In [ ]:
np.random.seed(42)
X_train = np.random.randn(2048, 8)
X_test = np.random.randn(512, 8)
y_train = np.sin(X_train).sum(axis=1, keepdims=True)
y_test = np.sin(X_test).sum(axis=1, keepdims=True)

train_loader = mm.DataLoader(X_train, y_train, batch_size=64, shuffle=True)
test_loader = mm.DataLoader(X_test, y_test, batch_size=64, shuffle=False)

model = mm.Sequential(
    mm.Linear(8, 32),
    mm.BatchNormalization(32),
    mm.ReLU(),
    mm.Linear(32, 64),
    mm.Dropout(0.25),
    mm.Sigmoid(),
    mm.Linear(64, 1)
)
optimizer = mm.Adam(model, lr=1e-2)
criterion = mm.MSELoss()

In [ ]:
from tqdm import tqdm

In [ ]:
num_epochs = 100
pbar = tqdm(range(1, num_epochs + 1))

for epoch in pbar:
    train_loss, test_loss = 0.0, 0.0

    model.train()
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)
        model.backward(X_batch, criterion.backward(predictions, y_batch))
        optimizer.step()

        train_loss += loss * X_batch.shape[0]

    model.eval()
    for X_batch, y_batch in test_loader:
        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)
        test_loss += loss * X_batch.shape[0]

    train_loss /= train_loader.num_samples()
    test_loss /= test_loader.num_samples()
    pbar.set_postfix({'train loss': train_loss, 'test loss': test_loss})

Final epoch | train loss: 0.265 | test loss: 0.305


## Result

The framework successfully optimizes the synthetic task, reaching a final test loss of approximately **0.305** after 100 epochs. This end-to-end check complements the implementation-level gradient and shape tests used during development.